# Candidate Generation: Popularity Baseline

This notebook fits a non-personalised popularity model, retrieves time-eligible Top-M candidates, writes the standard candidate format, and evaluates it with `cg_evaluation.py`.

Standard candidate format:

```text
impression_uid, cutoff, news_code
```

Each impression has exactly `CANDIDATE_CAP` rows, with `cutoff = 1..CANDIDATE_CAP`. `cutoff` defines nested Top-M membership; it is not a ranking metric.

Protocol:

- Fit popularity on `train` for validation and on `train + validation` for test.
- Order articles by prior click count, breaking ties by ascending `news_code`.
- Restrict candidates to `available_from <= timestamp`.
- Expand the timestamp-level popularity order into impression-level candidates.
- Evaluate Recall@M at M = 50, 100, 200 and 500.

The notebook runs the full pipeline by default. Set `FULL_RUN = False` in the
setup cell to score a request subset, write `*_smoke` artifacts and compute
subset recall instead of the shared evaluator.

## 1. Setup

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path
import sys

import numpy as np
import polars as pl

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'run_all.py').is_file()
)
sys.path.insert(0, str(ROOT / '5_Models' / '5_01_Candidate_Generation'))
from cg_evaluation import CandidateEvaluator
from cg_result_writer import CandidateWriter

PROCESSED = ROOT / '1_Data_Preparation' / 'processed'
FEATURES = ROOT / '4_Feature_Engineering' / 'artifacts' / 'prequential-all'
REPRESENTATIONS = ROOT / '3_Representations' / 'artifacts'
ARTIFACTS = ROOT / '5_Models' / '5_01_Candidate_Generation' / 'artifacts'
RESULTS = ROOT / 'results'

TARGET_SPLIT = 'validation'
FIT_SPLIT = 'train' if TARGET_SPLIT == 'validation' else 'train+validation'
FULL_RUN = True                 # Toggle here: False runs a smoke pipeline.
RUN_TAG = 'popularity' if FULL_RUN else 'popularity_smoke'
OUTPUT_DIR = ARTIFACTS / RUN_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
SMOKE_REQUESTS = 3_000
SEED = 4222
M_VALUES = (50, 100, 200, 500)
CANDIDATE_CAP = max(M_VALUES)

if TARGET_SPLIT not in {'validation', 'test'}:
    raise ValueError('TARGET_SPLIT must be validation or test')
if not FULL_RUN and SMOKE_REQUESTS < 1:
    raise ValueError('SMOKE_REQUESTS must be positive')

print('Repository:', ROOT)
print('Target split:', TARGET_SPLIT)
print('Fit split:', FIT_SPLIT)
print('Mode:', 'full' if FULL_RUN else 'smoke')
print('M values:', M_VALUES)
print('Candidate cap:', CANDIDATE_CAP)

## 2. Data ingestion and model-related processing

### Load catalogue, availability and target impressions

In [ ]:
catalogue = pl.read_parquet(
    PROCESSED / 'news_catalogue.parquet',
    columns=[
        'news_code',
        'exposures_train',
        'clicks_train',
        'exposures_validation',
        'clicks_validation',
    ],
)
availability = pl.read_parquet(
    REPRESENTATIONS / 'article_availability.parquet',
    columns=['news_code', 'available_from'],
)

In [ ]:
impressions = pl.read_parquet(
    PROCESSED / f'impressions_{TARGET_SPLIT}.parquet',
    columns=['impression_uid', 'user_code', 'timestamp'],
)

if not FULL_RUN:
    impressions = impressions.sort('timestamp', maintain_order=True)
    ordered_timestamps = impressions['timestamp'].to_numpy()
    starts = np.flatnonzero(
        np.r_[True, ordered_timestamps[1:] != ordered_timestamps[:-1]]
    )
    ends = np.r_[starts[1:], len(ordered_timestamps)]
    batches = list(zip(starts, ends))
    batch_rng = np.random.default_rng(SEED)
    selected_batches = []
    total = 0
    for index in batch_rng.permutation(len(batches)):
        start, end = batches[index]
        selected_batches.append((start, end))
        total += end - start
        if total >= SMOKE_REQUESTS:
            break
    selected_batches.sort()
    impressions = pl.concat(
        [
            impressions.slice(start, end - start)
            for start, end in selected_batches
        ]
    )
    if impressions.height == 0:
        raise ValueError('Smoke mode selected no requests')

## 3. Setting up the model

The popularity model is a non-personalised prior-exposure table. Section 4 fits and saves it; Section 5 applies it to time-eligible requests.

## 4. Training the model

### Fit and save the popularity table

The catalogue already stores split-level exposure and click counts, so there is no need to rescan the 70M training candidate rows. Those summaries are retrospective for their own split, which is safe because the entire fit split precedes the target split.

In [ ]:
prior_exposures = pl.col('exposures_train')
prior_clicks = pl.col('clicks_train')
if FIT_SPLIT == 'train+validation':
    prior_exposures = prior_exposures + pl.col('exposures_validation')
    prior_clicks = prior_clicks + pl.col('clicks_validation')

popularity = (
    catalogue.select(
        'news_code',
        prior_exposures.alias('prior_exposures'),
        prior_clicks.alias('prior_clicks'),
    )
    .with_columns(pl.col('prior_exposures').cast(pl.Int64).alias('popularity_score'))
    .join(availability, on='news_code', how='left')
    .sort(['popularity_score', 'news_code'], descending=[True, False])
)

if popularity.height != catalogue.height:
    raise ValueError('Popularity table lost catalogue rows')
if popularity['available_from'].null_count():
    raise ValueError('Missing available_from for some catalogue articles')
if popularity['popularity_score'].null_count():
    raise ValueError('Missing popularity scores')

model_path = OUTPUT_DIR / f"model_{FIT_SPLIT.replace('+', '_')}.parquet"
popularity.write_parquet(model_path)

print(f'Catalogue articles: {popularity.height:,}')
print(f'Prior clicks used for fitting: {popularity["popularity_score"].sum():,}')
print(f'Articles with at least one prior click: {popularity.filter(pl.col("popularity_score") > 0).height:,}')
print('Model saved to:', model_path)
popularity.head(10)

## 5. Results

### Build the timestamp-level popularity order

Popularity is identical for every user, so only eligibility changes between requests. Build the ordered eligible list once per timestamp, then expand it to the standard impression-level format.

In [ ]:
timestamps = impressions['timestamp'].unique().sort()
ts = timestamps.to_numpy()
news_codes = popularity['news_code'].to_numpy()
available_from = popularity['available_from'].to_numpy()

n_timestamps = len(ts)
candidate_news = np.empty((n_timestamps, CANDIDATE_CAP), dtype=np.int32)

for i, timestamp in enumerate(ts):
    eligible_rows = np.flatnonzero(available_from <= timestamp)
    if eligible_rows.size < CANDIDATE_CAP:
        raise ValueError(f'Only {eligible_rows.size} eligible articles at {timestamp}')
    candidate_news[i] = news_codes[eligible_rows[:CANDIDATE_CAP]]

print(f'Target impressions: {impressions.height:,}')
print(f'Unique timestamps: {n_timestamps:,}')
print(f'Candidates per timestamp: {CANDIDATE_CAP}')
print('Top 5 articles at first timestamp:', candidate_news[0, :5].tolist())

### Write the standard impression-level candidates

Each impression receives `CANDIDATE_CAP` rows. The write is batched so the full expansion does not need to sit in memory at once.

In [ ]:
recommendations_path = (
    OUTPUT_DIR / f'recommendations_{TARGET_SPLIT}_top{CANDIDATE_CAP}.parquet'
)
timestamp_to_index = {timestamp: i for i, timestamp in enumerate(ts)}
cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
writer = CandidateWriter(recommendations_path)
try:
    for batch in impressions.select('impression_uid', 'timestamp').iter_slices(
        n_rows=25_000
    ):
        batch_timestamps = batch['timestamp'].to_numpy()
        indices = np.fromiter(
            (timestamp_to_index[t] for t in batch_timestamps),
            dtype=np.int64,
            count=batch.height,
        )
        writer.add(
            np.repeat(batch['impression_uid'].to_numpy(), CANDIDATE_CAP),
            np.tile(cutoff_values, batch.height),
            candidate_news[indices].reshape(-1),
        )
finally:
    writer.close()

expected_rows = impressions.height * CANDIDATE_CAP
print(f'Expected candidate rows: {expected_rows:,}')
print('Candidates saved to:', recommendations_path)

### Check the saved candidate artifact

In [ ]:
saved_rows = pl.scan_parquet(recommendations_path).select(pl.len()).collect().item()
if saved_rows != expected_rows:
    raise ValueError(f'Expected {expected_rows:,} rows, found {saved_rows:,}')

print(f'Saved candidate rows: {saved_rows:,}')
print(pl.read_parquet(recommendations_path).head())

### Save the manifest

In [ ]:
def smoke_recall(recommendations_path, model_name, impression_uids):
    uid_frame = pl.DataFrame(
        {'impression_uid': np.asarray(impression_uids, dtype=np.int64)}
    )
    label_parts = sorted(
        (FEATURES / f'labels_{TARGET_SPLIT}').glob('*.parquet')
    )
    if not label_parts:
        raise FileNotFoundError(f'No labels for {TARGET_SPLIT}')
    labels = pl.read_parquet(
        label_parts,
        columns=['impression_uid', 'news_code', 'clicked'],
    ).join(uid_frame, on='impression_uid', how='inner')
    positives = labels.filter(pl.col('clicked') == 1).select(
        'impression_uid', 'news_code'
    )
    relevant = positives.group_by('impression_uid').agg(
        pl.len().alias('relevant')
    )
    base = uid_frame.join(
        relevant, on='impression_uid', how='left'
    ).with_columns(pl.col('relevant').fill_null(0).cast(pl.Int64))
    candidates = pl.read_parquet(
        recommendations_path,
        columns=['impression_uid', 'news_code', 'cutoff'],
    )
    rows = []
    for m in M_VALUES:
        selected = candidates.filter(pl.col('cutoff') <= m)
        hits = (
            positives.join(
                selected.select('impression_uid', 'news_code'),
                on=['impression_uid', 'news_code'],
                how='inner',
            )
            .group_by('impression_uid')
            .agg(pl.len().alias('hits'))
        )
        per_impression = base.join(
            hits, on='impression_uid', how='left'
        ).with_columns(pl.col('hits').fill_null(0).cast(pl.Int64))
        relevant_clicks = int(per_impression['relevant'].sum())
        retrieved_relevant = int(per_impression['hits'].sum())
        with_relevant = per_impression.filter(pl.col('relevant') > 0)
        recall_impression_mean = (
            float(
                (with_relevant['hits'] / with_relevant['relevant']).mean()
            )
            if with_relevant.height
            else float('nan')
        )
        hit_rate = (
            float((per_impression['hits'] > 0).sum() / per_impression.height)
            if per_impression.height
            else float('nan')
        )
        rows.append(
            {
                'model': model_name,
                'split': TARGET_SPLIT,
                'm': int(m),
                'recall_pooled': (
                    retrieved_relevant / relevant_clicks
                    if relevant_clicks
                    else float('nan')
                ),
                'recall_impression_mean': recall_impression_mean,
                'hit_rate': hit_rate,
                'impressions': int(per_impression.height),
                'relevant_clicks': relevant_clicks,
                'retrieved_relevant': retrieved_relevant,
                'mean_candidates': float(min(m, CANDIDATE_CAP)),
            }
        )
    return pl.DataFrame(rows)

In [ ]:
smoke_recall_path = None
if not FULL_RUN:
    smoke_evaluation = smoke_recall(
        recommendations_path,
        'popularity',
        impressions['impression_uid'].to_numpy(),
    )
    smoke_recall_path = OUTPUT_DIR / f'smoke_recall_{TARGET_SPLIT}.csv'
    smoke_evaluation.write_csv(smoke_recall_path)
    print(smoke_evaluation.sort('m'))
    print('Smoke recall:', smoke_recall_path)

In [ ]:
manifest = {
    'model': 'popularity',
    'score': 'prior_clicks',
    'tie_break': 'news_code ascending',
    'fit_split': FIT_SPLIT,
    'target_split': TARGET_SPLIT,
    'candidate_cap': CANDIDATE_CAP,
    'candidate_format': ['impression_uid', 'cutoff', 'news_code'],
    'eligibility': 'available_from <= timestamp',
    'catalogue_rows': popularity.height,
    'target_requests': impressions.height,
    'unique_timestamps': n_timestamps,
    'recommendation_rows': expected_rows,
    'mode': 'full' if FULL_RUN else 'smoke',
    'smoke_requests': None if FULL_RUN else SMOKE_REQUESTS,
    'model_file': model_path.name,
    'recommendations_file': recommendations_path.name,
    'smoke_recall_file': (
        smoke_recall_path.name if smoke_recall_path is not None else None
    ),
    'created_at': datetime.now(timezone.utc).isoformat(),
}
manifest_path = OUTPUT_DIR / f'manifest_{TARGET_SPLIT}.json'
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n')

print('Manifest saved to:', manifest_path)
manifest

### Evaluate Recall@M

Expected pooled recall is approximately 0.0406, 0.0925, 0.1367 and 0.2109 for M = 50, 100, 200 and 500.

In [ ]:
if FULL_RUN:
    evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
    evaluation = evaluator.evaluate(
        recommendations_path,
        model_name='popularity',
        m_values=M_VALUES,
    )

    evaluation_path = (
        RESULTS / f'candidate_generation_popularity_{TARGET_SPLIT}.csv'
    )
    evaluation.write_csv(evaluation_path)
    print('Evaluation saved to:', evaluation_path)
    evaluation
else:
    print('Smoke run: shared evaluator skipped; subset recall written.')

## Notes and limitations

- It is non-personalised: every user receives the same popularity order apart from eligibility.
- Articles never clicked in the fit split receive score 0. Roughly 14.4% of validation clicks are on such articles, so this baseline cannot retrieve them.
- It is frozen per split: popularity is not updated within the target window. This is intentional for the first baseline.
- `available_from` is an observed-availability proxy, not a verified publication timestamp. Eligibility follows the project-wide convention.
- The output is standard impression-level candidates, ready for the re-ranker and any candidate-generation evaluator.